# Grouped (duplicate-aware) evaluation — separate new experiment

**Question.** Do the main C2 degradation and C3 recovery hold when an exact
duplicate feature vector can never appear in both the training and the test
data?

30.6% of the prepared rows belong to exact-duplicate groups, and on average
28.5% of random-split test rows have an exact copy in training. This run keeps
every duplicate group on one side of the split. Everything else follows the
locked main pipeline: same prepared data, same historical hybrid768
transformation, training-only KS/Holm and MI selection, same model settings.

- **Groups:** one per exact 70-feature vector (89,064 groups).
- **Label conflicts:** kept whole in their own stratum; no rows dropped.
- **Split:** 70:30 per class, ten seeds (0–9), groups never cross.
- **Fits:** 60 (C1 and C2 share one model; C3 is one model). About 35–45 minutes.
- **Statistics:** descriptive, plus the unchanged registered weighted-F1
  procedure (paired tests and TOST ±0.01 / ±0.005, Holm across models).
- **Decision rule (fixed now):** the main claim holds for a model if C2 is
  below C1 (Holm p < 0.05) and C3 − C1 meets TOST ±0.01 after Holm.

The full fixed plan is `environment/grouped_evaluation_plan.json`. The
notebook refuses a package whose runner or plan differs.

## Steps

1. Select **Runtime → Run all**.
2. Upload the **PQC_IDS_Reproduction_Working_Copy.zip supplied with this notebook**.
3. Connect the Drive that holds the four original CSV files
   (`/content/drive/MyDrive/cicids2018`; change `DATA_DIR` only if they moved).
4. Return **PQC_IDS_Grouped_Evaluation_Results.zip**, even if it records a failure.
   A copy is saved to `MyDrive/pqc_ids_grouped_evaluation/<run_id>/`. If the
   ZIP is over 30 MB, use the same split-into-parts cell as before.

Prepared with assistant assistance. Locked results are not modified.


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import stat
import subprocess
import sys
import tempfile
import traceback
import uuid
import zipfile
from google.colab import files, drive

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
SESSION = Path(tempfile.mkdtemp(prefix="pqc_grouped_evaluation_", dir="/content"))

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


# Edit this folder only if the four original CSV files moved.
DATA_DIR = Path("/content/drive/MyDrive/cicids2018")
DRIVE_OUTPUT = SESSION / "records"
DRIVE_OUTPUT.mkdir()
RUN_OUTPUT = DRIVE_OUTPUT / "results"
SUMMARY = {"run_id": RUN_ID, "status": "preparing", "fits_completed": 0,
           "scope": "Separate grouped (duplicate-aware) evaluation, preserving historical records",
           "historical_results_modified": False, "colab_kernel_python": platform.python_version()}
DRIVE_CONNECTED = False

def save_summary():
    (DRIVE_OUTPUT / "notebook_status.json").write_text(json.dumps(SUMMARY, indent=2) + "\n")

save_summary()
try:
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Upload exactly one updated working-copy ZIP.")
    archive_path = SESSION / "working_copy.zip"
    archive_path.write_bytes(uploaded[zip_names[0]])
    del uploaded
    extracted = SESSION / "package"
    with zipfile.ZipFile(archive_path) as archive:
        if archive.testzip() is not None:
            raise ValueError("The ZIP failed its integrity check. Download the working copy again.")
        for info in archive.infolist():
            path = PurePosixPath(info.filename)
            if path.is_absolute() or ".." in path.parts or stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError("Unexpected archive path: " + info.filename)
        archive.extractall(extracted)
    manifest_paths = list(extracted.glob("*/package_manifest.json"))
    if len(manifest_paths) != 1:
        raise ValueError("Expected one working-copy package manifest.")
    PACKAGE = manifest_paths[0].parent
    manifest = json.loads(manifest_paths[0].read_text())
    listed = set()
    for item in manifest["files"]:
        path = (PACKAGE / item["path"]).resolve()
        if not path.is_relative_to(PACKAGE.resolve()):
            raise ValueError("Invalid manifest path.")
        if not path.is_file() or path.stat().st_size != item["bytes"] or sha256(path) != item["sha256"]:
            raise ValueError("Package file does not match the manifest: " + item["path"])
        listed.add(item["path"])
    actual = {p.relative_to(PACKAGE).as_posix() for p in PACKAGE.rglob("*") if p.is_file()}
    if actual != listed | {"package_manifest.json"} or len(listed) != len(manifest["files"]):
        raise ValueError("The archive contains unlisted files or duplicate manifest records.")
    if not (PACKAGE / "scripts/run_grouped_evaluation.py").is_file():
        raise ValueError("This is an older working copy. Use the ZIP provided with this notebook.")
    # The grouped-evaluation runner and its fixed plan must match exactly.
    EXPECTED_RUNNER_SHA256 = "6fe4628605d98a4b8fa7f402773ab0cb63b877e6dba8d262aaee2bf2ab3d6854"
    EXPECTED_PLAN_SHA256 = "7a48a053c8d61b20d44a1757b6a0eb95cabb4c61c9a3bf98af031dc9f44cb0fc"
    RUNNER_SHA256 = sha256(PACKAGE / "scripts/run_grouped_evaluation.py")
    if RUNNER_SHA256 != EXPECTED_RUNNER_SHA256:
        raise ValueError("This working copy has a different grouped-evaluation runner. Use the ZIP supplied with this notebook.")
    if sha256(PACKAGE / "environment/grouped_evaluation_plan.json") != EXPECTED_PLAN_SHA256:
        raise ValueError("The grouped-evaluation plan differs from the fixed version.")
    print("Grouped-evaluation runner and plan verified:", RUNNER_SHA256)
    print("Working-copy files verified:", len(listed))
    print("Run identifier:", RUN_ID)

    SUMMARY["package_sha256"] = sha256(archive_path)
    SUMMARY["runner_sha256"] = RUNNER_SHA256
    SUMMARY["package_manifest_sha256"] = sha256(PACKAGE / "package_manifest.json")
    shutil.copy2(PACKAGE / "package_manifest.json", DRIVE_OUTPUT / "working_copy_manifest.json")
    for name in ("main_known_versions.json", "main_requirements.txt", "grouped_evaluation_plan.json"):
        shutil.copy2(PACKAGE / "environment" / name, DRIVE_OUTPUT / name)
    TARGET = json.loads((PACKAGE / "environment/main_known_versions.json").read_text())
    drive.mount("/content/drive")
    DRIVE_CONNECTED = True
    SUMMARY["data_directory"] = str(DATA_DIR)
    SUMMARY["target"] = TARGET
    save_summary()

    RUN_ENV = os.environ.copy()
    for name in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV"):
        RUN_ENV.pop(name, None)
    RUN_ENV["PYTHONNOUSERSITE"] = "1"
    RUN_ENV["UV_PYTHON_INSTALL_DIR"] = str(SESSION / "python_runtimes")
    RUN_ENV["UV_CACHE_DIR"] = str(SESSION / "uv_cache")

    def run_logged(command, log_name, env):
        with (DRIVE_OUTPUT / log_name).open("w", encoding="utf-8") as log:
            log.write("Command: " + json.dumps([str(s) for s in command]) + "\n")
            log.flush()
            with subprocess.Popen([str(s) for s in command], stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                                  text=True, bufsize=1, env=env) as process:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                return process.wait()

    ENVIRONMENT_READY = False
    VENV = SESSION / "grouped_evaluation_environment"
    FIT_PYTHON = VENV / "bin/python"
    try:
        tooling = SESSION / "uv_tooling"
        bootstrap = [sys.executable, "-m", "pip", "install", "--target", tooling,
                     "--no-deps", "uv==0.12.15"]
        if run_logged(bootstrap, "01_uv_install.log", RUN_ENV) != 0:
            raise RuntimeError("Could not install the pinned uv tool.")
        uv_env = RUN_ENV.copy()
        uv_env["PYTHONPATH"] = str(tooling)
        uv = [sys.executable, "-m", "uv"]
        create = uv + ["venv", "--python", TARGET["python"], "--managed-python", VENV]
        if run_logged(create, "02_python_setup.log", uv_env) != 0:
            raise RuntimeError("Could not create the requested Python runtime.")
        install = uv + ["pip", "install", "--python", FIT_PYTHON, "--only-binary", ":all:",
                        "-r", PACKAGE / "environment/main_requirements.txt"]
        if run_logged(install, "03_packages_install.log", uv_env) != 0:
            raise RuntimeError("Could not install the declared package versions.")
        ENVIRONMENT_READY = True
        SUMMARY["status"] = "environment_installed"
    except Exception as error:
        SUMMARY["status"] = "environment_setup_failed"
        SUMMARY["error"] = str(error)
        (DRIVE_OUTPUT / "setup_traceback.txt").write_text(traceback.format_exc())
        print("Setup failed:", error)
    finally:
        save_summary()
    print("Environment ready:", ENVIRONMENT_READY)

    if not ENVIRONMENT_READY:
        raise RuntimeError("Grouped evaluation did not run because environment setup failed; return the exported logs.")
    SUMMARY["status"] = "running_grouped_evaluation"
    save_summary()
    returncode = run_logged([FIT_PYTHON, "-u", PACKAGE / "scripts/run_grouped_evaluation.py",
                             "--data-dir", DATA_DIR, "--output", RUN_OUTPUT,
                             "--cache-dir", SESSION / "input_cache"],
                            "04_grouped_evaluation.log", RUN_ENV)
    SUMMARY["runner_returncode"] = returncode
    status_path = RUN_OUTPUT / "grouped_status.json"
    if not status_path.is_file():
        raise RuntimeError("Grouped runner produced no status record; return the ZIP.")
    audit_status = json.loads(status_path.read_text())
    SUMMARY["grouped_status"] = audit_status["status"]
    SUMMARY["statistics_status"] = audit_status.get("statistics_status")
    SUMMARY["fits_completed"] = audit_status.get("fits_completed", 0)
    if returncode != 0 or audit_status["status"] != "grouped_completed" or audit_status.get("metric_rows") != 90 or audit_status.get("completed_splits") != list(range(10)):
        raise RuntimeError("Grouped evaluation did not complete; return the ZIP containing its failure evidence.")
    SUMMARY["status"] = "grouped_completed"
    print("Completed fits:", audit_status["fits_completed"], "(planned 60)")
    print("Metric and prediction records:", audit_status["metric_rows"])
    print("Return the ZIP for independent interpretation; no historical scores were replaced.")

except BaseException as error:
    SUMMARY["status"] = "grouped_incomplete"
    SUMMARY["error"] = str(error)
    (DRIVE_OUTPUT / "notebook_traceback.txt").write_text(traceback.format_exc())
    print("Please return the ZIP with this recorded issue:", error)
finally:
    SUMMARY["exported_utc"] = datetime.now(timezone.utc).isoformat()
    save_summary()
    result_files = sorted(p for p in DRIVE_OUTPUT.rglob("*") if p.is_file())
    items = [{"path": p.relative_to(DRIVE_OUTPUT).as_posix(), "bytes": p.stat().st_size,
              "sha256": sha256(p)} for p in result_files]
    manifest_path = DRIVE_OUTPUT / "result_manifest.json"
    manifest_path.write_text(json.dumps({"run_id": RUN_ID, "files": items}, indent=2) + "\n")
    result_files.append(manifest_path)
    result_zip = SESSION / "PQC_IDS_Grouped_Evaluation_Results.zip"
    with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in result_files:
            archive.write(path, path.relative_to(DRIVE_OUTPUT).as_posix())
    with zipfile.ZipFile(result_zip) as archive:
        if archive.testzip() is not None:
            raise RuntimeError("Result archive failed its integrity check.")
    runner_status = RUN_OUTPUT / "grouped_status.json"
    if runner_status.is_file():
        captured = json.loads(runner_status.read_text())
        print("Runner recorded fits:", captured.get("fits_completed", 0))
    if DRIVE_CONNECTED:
        try:
            saved = Path("/content/drive/MyDrive/pqc_ids_grouped_evaluation") / RUN_ID
            saved.mkdir(parents=True, exist_ok=False)
            shutil.copy2(result_zip, saved / result_zip.name)
            print("Drive copy:", saved / result_zip.name)
        except OSError as error:
            print("Drive copy failed, but the downloadable ZIP is available:", error)
    print("Final status:", SUMMARY["status"])
    print("Fits recorded:", SUMMARY.get("fits_completed", "see runner status"))
    print("ZIP SHA-256:", sha256(result_zip))
    files.download(str(result_zip))
